In [1]:
import json
import glob
import re
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
# Load all results_*.json files
all_records = []
results = ["results_1776605286.json"]
# results = glob.glob("results_*.json")
for filepath in sorted(results):
    with open(filepath) as f:
        data = json.load(f)
        all_records.extend(data)
        print(f"{filepath}: {len(data)} records")
print(f"Total: {len(all_records)} records")

In [ ]:
def shorten_query(q):
    """Create a short readable label from a query string."""
    q = re.sub(r"\s+INTO\s+\w+", "", q, flags=re.IGNORECASE)
    q = re.sub(r'VARSIZED\("([^"]+)"\)', r'"\1"', q)
    q = re.sub(r'INT32\((\d+)\)', r'\1', q)
    q = re.sub(r'FLOAT64\(([\d.]+)\)', r'\1', q)
    q = re.sub(r'\s+', ' ', q).strip()
    return q

# File sizes for throughput calculation
BID_FILE_SIZE = 6678514674   # bytes
PERSON_FILE_SIZE = 880941057 # bytes

def get_file_size(query):
    """Return file size based on source table used in the query."""
    q_lower = query.lower()
    if "bid" in q_lower:
        return BID_FILE_SIZE
    return PERSON_FILE_SIZE

# Build dataframe with execution time and throughput
rows = []
for r in all_records:
    if r.get("status") != "Stopped" or not r.get("started") or not r.get("stopped"):
        continue
    started = datetime.strptime(r["started"], "%Y-%m-%d %H:%M:%S.%f")
    stopped = datetime.strptime(r["stopped"], "%Y-%m-%d %H:%M:%S.%f")
    exec_time = (stopped - started).total_seconds()
    query = shorten_query(r["query"].strip())
    file_size = get_file_size(query)
    throughput = file_size / exec_time  # bytes/s
    rows.append({
        "query": query,
        "threads": r["threads_number"],
        "buffer_size": r["buffer_size"],
        "exec_time_s": exec_time,
        "throughput_MBs": throughput / (1024 * 1024),  # MB/s
    })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
print(f"Thread counts: {sorted(df['threads'].unique())}")
print(f"Buffer sizes: {sorted(df['buffer_size'].unique())}")
print(f"Queries ({len(df['query'].unique())}):")
for q in sorted(df['query'].unique()):
    src = "bid" if "bid" in q.lower() else "person"
    print(f"  [{src}] {q}")
df.head()

In [ ]:
# Median throughput across all queries, x=buffer_size, lines=threads
def fmt_buf(x, _):
    if x >= 1048576:
        return f"{int(x/1048576)}M"
    if x >= 1024:
        return f"{int(x/1024)}K"
    return str(int(x))

thread_counts = sorted(df["threads"].unique())
cmap = plt.cm.get_cmap("tab10", len(thread_counts))
colors = {t: cmap(i) for i, t in enumerate(thread_counts)}

fig, ax = plt.subplots(figsize=(14, 10))

for t in thread_counts:
    t_data = df[df["threads"] == t]
    grouped = t_data.groupby("buffer_size")["throughput_MBs"].median().sort_index()
    ax.plot(grouped.index, grouped.values, marker="o",
            label=f"{t} threads", color=colors[t],
            linewidth=1.5, markersize=5)

ax.set_xscale("log", base=2)
ax.set_xlabel("Operator Buffer Size")
ax.set_ylabel("Throughput (MB/s)")
ax.set_title("Median Throughput vs. Buffer Size (by thread count)")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_buf))
ax.grid(True, alpha=0.3)
ax.legend(title="Threads", fontsize=8, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()

In [ ]:
# Summary table: median throughput (MB/s) per buffer_size x threads
summary = df.groupby(["buffer_size", "threads"])["throughput_MBs"].median().unstack()
summary.columns = [f"{int(c)} threads" for c in summary.columns]
summary.index = [fmt_buf(i, None) for i in summary.index]
summary.round(2)